# DSC0152 — notebook v70

## Kenapa v70 ada

Public terbaik tetap **0.56424** dari `v66_04` (`ext0.5 | tm30 | lgb_lean`). `v69_10`
menguji arah yang sama tetapi mengganti pivot `lgb_lean` dengan `extra`, dan
hasilnya turun ke **0.55989**. Pelajarannya: pivot dan resep ekstrapolasi v66
dipertahankan; yang boleh berubah adalah **kualitas core** yang diekstrapolasi.

v70 karena itu hanya mengubah dua hal:

1. **Fitur lintas sumber baru** — panel bulanan repayment × cash flow × e-commerce,
   status pinjaman, dan kedalaman e-commerce. Semua dihitung pada `decision_datetime`
   masing-masing aplikasi, jadi tidak ada leakage.
2. **Training lebih banyak** — 15 fold, 10 fold seed, 2 varian LightGBM. Leg v66
   tidak dilatih ulang; prediksi OOF/test-nya dipakai lagi apa adanya, sehingga
   arah public yang sudah terbukti tidak ikut berubah.

Hasil utama: leg `v70_extra` bagged **AUC 0.848369 / KS 0.528503**, di atas
`v69_extra` (0.848132 / 0.527736). Kandidat ekstrapolasi terbaik
`plateau|tm30w7|v70_extra|lgb_lean` mencapai **KS OOF 0.530232** dengan
Spearman 0.9984 terhadap `v66_04`.

In [1]:
import json, os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata, ks_2samp
from sklearn.metrics import roc_auc_score

RUN_TRAINING = False
FOLDS = 15
FOLD_SEEDS = '42,7,13,21,5,101,3,17,11,23'
LGB_SEEDS = '0'
VARIANTS = 'extra,stiff7'
DROP_PREFIXES = 'cf_,pv_'

HERE = Path.cwd().resolve()
ROOT = HERE
for cand in [HERE, *HERE.parents]:
    if (cand / 'data').is_dir() and (cand / 'agas').is_dir():
        ROOT = cand
        break
AGAS = ROOT / 'agas'
VER = AGAS / 'v70'
OUT = VER / 'output'
TRAINING = VER / 'training'
sys.path.insert(0, str(AGAS / 'shared'))
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print('repo  :', ROOT)
print('version:', VER)
print('output :', OUT)
print('train  :', TRAINING)
assert (ROOT / 'data/train_applications.csv').exists()

repo  : /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026
version: /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v70
output : /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v70/output
train  : /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v70/training


## 1. Probe sumber sinyal

Sebelum menambah fitur, tiap sumber diuji sendirian dengan 5 fold dan seed 42.
Tujuannya hanya memastikan blok baru punya sinyal mentah, bukan untuk memilih
model final.

In [2]:
z = np.load(OUT / 'source_auc_v70.npz', allow_pickle=True)
y = z['y']
rows = []
for k in z.files:
    if not k.startswith('oof_'):
        continue
    p = z[k]
    rows.append({'source': k[4:], 'auc': roc_auc_score(y, p),
                 'ks': ks_2samp(p[y == 1], p[y == 0]).statistic})
src = pd.DataFrame(rows).sort_values('auc', ascending=False)
display(src.style.format({'auc': '{:.4f}', 'ks': '{:.4f}'}))

# per-source rank blend: does simple averaging recover the all-feature model?
names = [k for k in z.files if k.startswith('oof_') and k != 'oof_all']
blend = np.mean([rankdata(z[k]) / len(y) for k in names], axis=0)
print(f"rank blend of {len(names)} sources: AUC {roc_auc_score(y, blend):.4f} "
      f"(all-feature model {roc_auc_score(y, z['oof_all']):.4f})")

,source,auc,ks
6,all,0.8459,0.5242
5,app_profile,0.8308,0.5020
3,ecommerce,0.7635,0.3906
1,repayment,0.7462,0.3397
0,bureau,0.6869,0.2659
4,previous,0.6309,0.1888
2,account,0.6207,0.1761


rank blend of 6 sources: AUC 0.8138 (all-feature model 0.8459)


## 2. Blok fitur baru dan screening

Empat blok dibangun di `agas/v70/training/feat_v70.py`:

| prefix | isi | jumlah |
|---|---|---:|
| `ms_` | panel bulanan repayment (due/late/unpaid) × cash flow (net, saldo, low-balance days) | 67 |
| `ls_` | status pinjaman berjalan: late/dpd per pinjaman, aktif vs lunas, rasio bayar | 40 |
| `ed_` | kedalaman e-commerce: keranjang, kategori, tren harga, konsentrasi | 51 |
| `ss_` | panel 3 arah: late × net cash flow × e-commerce spend | 42 |

Screening memakai resep yang sama untuk semua baris: 5 fold, fold seed
42/7/13, satu LGB seed, config `extra`, factor features aktif. Yang berubah
hanya blok yang dimasukkan.

In [3]:
rows = []
labels = {
    's0_base': 'baseline (v69 lean + factor)',
    's1_ms': '+ monthstress',
    's2_ls': '+ loanstatus',
    's3_ed': '+ ecomdepth',
    's4_all': '+ ms + ls + ed',
    's5_ss': '+ spendstress',
    's6_allss': '+ all + spendstress',
}
for key, label in labels.items():
    d = json.loads((OUT / f'summary_{key}.json').read_text())
    rows.append({'run': key, 'blok': label, 'n_lean': d['n_lean'],
                 'auc': d['mean_auc'], 'ks': d['mean_ks'],
                 'auc_sd': d['sd_auc'], 'menit': d['minutes']})
scr = pd.DataFrame(rows)
display(scr.style.format({'auc': '{:.6f}', 'ks': '{:.6f}',
                          'auc_sd': '{:.6f}', 'menit': '{:.1f}'}))

base = scr.loc[scr.run == 's0_base'].iloc[0]
scr['d_auc'] = scr.auc - base.auc
scr['d_ks'] = scr.ks - base.ks
print(scr[['run', 'd_auc', 'd_ks']].to_string(index=False))
print('\nKesimpulan: ms_ konsisten positif; ls_ netral; ed_ negatif sendirian '
      'tetapi +all sedikit di atas +ms; ss_ negatif dan dibuang.')

,run,blok,n_lean,auc,ks,auc_sd,menit
0,s0_base,baseline (v69 lean + factor),526,0.847013,0.526286,0.000212,9.9
1,s1_ms,+ monthstress,593,0.847135,0.527648,0.000228,10.7
2,s2_ls,+ loanstatus,566,0.847065,0.526527,0.000219,10.4
3,s3_ed,+ ecomdepth,577,0.846887,0.526514,0.000249,10.8
4,s4_all,+ ms + ls + ed,684,0.847219,0.527724,0.000183,11.5
5,s5_ss,+ spendstress,568,0.846972,0.526633,0.000257,8.7
6,s6_allss,+ all + spendstress,726,0.847023,0.526422,0.000134,16.0


     run     d_auc     d_ks
 s0_base  0.000000 0.000000
   s1_ms  0.000121 0.001361
   s2_ls  0.000052 0.000240
   s3_ed -0.000126 0.000227
  s4_all  0.000206 0.001438
   s5_ss -0.000042 0.000347
s6_allss  0.000009 0.000135

Kesimpulan: ms_ konsisten positif; ls_ netral; ed_ negatif sendirian tetapi +all sedikit di atas +ms; ss_ negatif dan dibuang.


## 3. Training final

Leg v66 (`lgb_full`, `lgb_lean`, `lgbx_full`, `cat_full`, `xgb_full`, `linear`)
dipakai ulang dari `agas/v66/output/preds_v66.npz`. Yang baru hanya dua varian
`lgb_lean` v70 di atas feature set `+all` (684 kolom lean, 690 dengan factor):

- **extra** — `num_leaves=15`, `feature_fraction=0.45`, `extra_trees=True`
- **stiff7** — `num_leaves=7`, `feature_fraction=0.20`, `lambda_l2=40`

Konfigurasi: 15 fold, fold seed `42,7,13,21,5,101,3,17,11,23`, LGB seed 0.

In [4]:
cmd = (f"python3 {TRAINING}/train_lean_v70.py "
       f"--features {OUT}/features_v70.pkl --variants {VARIANTS} "
       f"--folds {FOLDS} --fold-seeds {FOLD_SEEDS} --lgb-seeds {LGB_SEEDS} "
       f"--drop-prefixes {DROP_PREFIXES} --out-prefix v70")
print(cmd)
if RUN_TRAINING:
    subprocess.run(cmd, shell=True, check=True, cwd=ROOT)
else:
    print('\nRUN_TRAINING=False: memakai preds_v70.npz yang sudah ada.')
assert (OUT / 'preds_v70.npz').exists(), 'preds_v70.npz belum ada; jalankan training dulu'

python3 /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v70/training/train_lean_v70.py --features /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v70/output/features_v70.pkl --variants extra,stiff7 --folds 15 --fold-seeds 42,7,13,21,5,101,3,17,11,23 --lgb-seeds 0 --drop-prefixes cf_,pv_ --out-prefix v70

RUN_TRAINING=False: memakai preds_v70.npz yang sudah ada.


In [5]:
z70 = np.load(OUT / 'preds_v70.npz', allow_pickle=True)
y70 = z70['y'].astype(int)
per_seed = json.loads((OUT / 'summary_v70.json').read_text())['per_seed']

rows = []
for seed, d in per_seed.items():
    for v, m in d.items():
        rows.append({'fold_seed': int(seed), 'variant': v,
                     'auc': m['auc'], 'ks': m['ks']})
seeds = pd.DataFrame(rows)
display(seeds.pivot(index='fold_seed', columns='variant', values='auc')
        .style.format('{:.6f}'))

summary = json.loads((OUT / 'summary_v70.json').read_text())
bag = pd.DataFrame([
    {'variant': v, 'auc': summary[v]['auc'], 'ks': summary[v]['ks'],
     'n_seeds': len(summary['fold_seeds']), 'folds': summary['folds']}
    for v in summary['variants']
])
display(bag.style.format({'auc': '{:.6f}', 'ks': '{:.6f}'}))
print('v69_extra: AUC 0.848132 / KS 0.527736')
print(f"v70_extra: AUC {summary['extra']['auc']:.6f} / KS {summary['extra']['ks']:.6f}")
print(f"delta     : AUC {summary['extra']['auc'] - 0.848132:+.6f} / "
      f"KS {summary['extra']['ks'] - 0.527736:+.6f}")

variant,extra,stiff7
fold_seed,,
3,0.848124,0.847844
5,0.847984,0.847247
7,0.847880,0.847631
11,0.848083,0.847466
13,0.847907,0.847593
17,0.847923,0.847252
21,0.847941,0.847500
23,0.847945,0.847547
42,0.847821,0.847238


,variant,auc,ks,n_seeds,folds
0,extra,0.848369,0.528503,10,15
1,stiff7,0.847843,0.527944,10,15


v69_extra: AUC 0.848132 / KS 0.527736
v70_extra: AUC 0.848369 / KS 0.528503
delta     : AUC +0.000237 / KS +0.000767


## 4. Rantai blend dan kandidat

Rantai v66 tidak diubah:

```
trees4 = mean(rank(lgb_full, lgb_lean, cat_full, xgb_full))
core   = (2 * trees4 + rank(lgbx_full)) / 3
tm30   = 0.7 * core + 0.3 * rank(linear)
ext_r  = rank01(tm30 + r * (tm30 - rank(lgb_lean)))
```

v70 menambahkan leg baru ke `trees`, dan menguji bobot `w` yang lebih besar
untuk leg baru (`tm30w3/5/7`). Ekstrapolasi tetap terhadap `lgb_lean` v66.
Pemilihan kandidat memakai AUC (bukan KS mentah), lantai AUC 0.0025 di bawah
kandidat terbaik, dan deduplikasi Spearman 0.9995.

In [6]:
cand = json.loads((OUT / 'summary_v70_candidates.json').read_text())
top = pd.DataFrame(cand['candidates'])
cols = ['slot', 'name', 'auc', 'ks', 'ks_minus_sigma', 'spearman_v66_04', 'valid']
display(top[cols].head(12).style.format({
    'auc': '{:.6f}', 'ks': '{:.6f}', 'ks_minus_sigma': '{:.6f}',
    'spearman_v66_04': '{:.6f}'}))
print(f"pool={cand['pool_size']} eligible={cand['eligible']} kept={cand['kept']}")
print('best v70 leg:', cand['best_v70_leg'])

,slot,name,auc,ks,ks_minus_sigma,spearman_v66_04,valid
0,1,leg|v70_extra,0.848369,0.528503,0.524903,0.989875,True
1,2,leg|x69_extra,0.848132,0.527736,0.524136,0.991086,True
2,3,tm30w7|v70_extra,0.847852,0.528663,0.525063,0.998486,True
3,4,leg|v70_stiff7,0.847843,0.527944,0.524344,0.990402,True
4,5,leg|x69_stiff7,0.847653,0.527933,0.524333,0.991371,True
5,6,plateau|tm30w7|v70_extra|lgb_lean,0.847623,0.530232,0.526632,0.998421,True
6,7,ext0.5|tm30v70|lgb_lean,0.847249,0.529397,0.525797,0.999670,True
7,8,tm30v66,0.847223,0.528476,0.524876,0.999216,True
8,9,ext0.25|tm30w7|v70_stiff7|x69_extra,0.847164,0.528747,0.525147,0.999052,True
9,10,trees4,0.847142,0.525693,0.522093,0.995111,True


pool=195 eligible=143 kept=21
best v70 leg: v70_extra


## 5. Validasi CSV

Semua file harus lolos empat pemeriksaan: 30.000 baris, `application_id` unik
dan persis sama dengan `data/test_applications.csv`, probabilitas di [0, 1],
tidak ada NaN.

In [7]:
test_ids = set(pd.read_csv(ROOT / 'data/test_applications.csv',
                           usecols=['application_id']).application_id)
rows = []
for p in sorted((OUT).glob('submission_v70_*.csv')):
    d = pd.read_csv(p)
    ok = (len(d) == 30000 and d.application_id.is_unique
          and set(d.application_id) == test_ids
          and d.default_probability.notna().all()
          and d.default_probability.between(0, 1).all())
    rows.append({'file': p.name, 'rows': len(d), 'valid': bool(ok),
                 'min': d.default_probability.min(),
                 'max': d.default_probability.max()})
val = pd.DataFrame(rows)
print(f'{len(val)} file, {val.valid.sum()} valid, {(~val.valid).sum()} invalid')
display(val.head(25).style.format({'min': '{:.4f}', 'max': '{:.4f}'}))
assert val.valid.all()

21 file, 21 valid, 0 invalid


,file,rows,valid,min,max
0,submission_v70_01_leg_v70_extra.csv,30000,True,0.0000,1.0000
1,submission_v70_02_leg_x69_extra.csv,30000,True,0.0000,1.0000
2,submission_v70_03_tm30w7_v70_extra.csv,30000,True,0.0000,1.0000
3,submission_v70_04_leg_v70_stiff7.csv,30000,True,0.0000,1.0000
4,submission_v70_05_leg_x69_stiff7.csv,30000,True,0.0000,1.0000
5,submission_v70_06_plateau_tm30w7_v70_extra_lgb_lean.csv,30000,True,0.0000,1.0000
6,submission_v70_07_ext0.5_tm30v70_lgb_lean.csv,30000,True,0.0000,1.0000
7,submission_v70_08_tm30v66.csv,30000,True,0.0000,1.0000
8,submission_v70_09_ext0.25_tm30w7_v70_stiff7_x69_extra.csv,30000,True,0.0000,1.0000
9,submission_v70_10_trees4.csv,30000,True,0.0000,1.0000


## 6. Rekomendasi submit

Tiga kandidat yang paling masuk akal, berurutan:

1. **`submission_v70_06_plateau_tm30w7_v70_extra_lgb_lean.csv`** — pilihan utama.
   KS OOF tertinggi (0.530232), AUC 0.847623, masih keluarga ekstrapolasi v66
   (Spearman 0.9984), dan rasio 0.25/0.5/0.75 dirata-rata sehingga tidak
   bergantung pada satu rasio yang kebetulan bagus.
2. **`submission_v70_07_ext0.5_tm30v70_lgb_lean.csv`** — pilihan konservatif.
   Paling dekat dengan resep public 0.56424 (Spearman 0.99967), memakai kedua
   varian v70 dengan bobot sama.
3. **`submission_v70_01_leg_v70_extra.csv`** — kontrol AUC. Model mentah
   terkuat (AUC 0.848369) tanpa ekstrapolasi; arah yang belum pernah diuji di
   public.

Kalau hanya boleh satu: **slot 06**. Kalau boleh dua: **06 + 07**.

In [8]:
best = {c['slot']: c for c in cand['candidates']}
for slot in (6, 7, 1):
    c = best[slot]
    print(f"slot {slot:02d}  {c['file']}")
    print(f"         AUC {c['auc']:.6f}  KS {c['ks']:.6f}  "
          f"Spearman vs v66_04 {c['spearman_v66_04']:.6f}")

slot 06  submission_v70_06_plateau_tm30w7_v70_extra_lgb_lean.csv
         AUC 0.847623  KS 0.530232  Spearman vs v66_04 0.998421
slot 07  submission_v70_07_ext0.5_tm30v70_lgb_lean.csv
         AUC 0.847249  KS 0.529397  Spearman vs v66_04 0.999670
slot 01  submission_v70_01_leg_v70_extra.csv
         AUC 0.848369  KS 0.528503  Spearman vs v66_04 0.989875
